In [42]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(
        **params
    )
    return message.content[0].text

# Start with an empty message list
messages = []

# # Add the initial user question
# add_user_message(messages, "Define quantum computing in one sentence")

# # Get Claude's response
# answer = chat(messages)

# # Add Claude's response to the conversation history
# add_assistant_message(messages, answer)

# print(messages)

# # Add a follow-up question
# add_user_message(messages, "Write another sentence")

# # Get the follow-up response with full context
# final_answer = chat(messages)

while True:
    input_text = input("You: ")
    add_user_message(messages, input_text)
    if not input_text.strip():
        break
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print("Claude:", answer)


In [25]:
messages = []

system = """
Your are a patient math tutor.
Do not directly answer user questions.
Guide the user to the solution step by step
"""

add_user_message(messages, "What is 5x + 3 = 2 for x?")
answer = chat(messages, system)
add_assistant_message(messages, answer)
print("Claude:", answer)

Claude: I'll help you solve for x step by step!

You have the equation: **5x + 3 = 2**

The goal is to isolate x on one side of the equation. 

**First step**: What do you think we should do to get rid of the "+ 3" on the left side? 

Remember, whatever operation we perform, we need to do it to *both* sides of the equation to keep it balanced.


In [26]:
messages = []

system = """
You are a programmer obsessed with writing code that is as concise as possible.

yet you give a great value to code readability and name variables and functions with fully descriptive names.
"""

add_user_message(messages, "Write a python function that checks string for duplicate characters")
answer = chat(messages, system)
add_assistant_message(messages, answer)
print("Claude:", answer)

Claude: ```python
def has_duplicate_characters(string: str) -> bool:
    return len(string) != len(set(string))
```

This function converts the string to a set (which removes duplicates) and compares lengths. If they differ, duplicates exist.

**Usage:**
```python
has_duplicate_characters("hello")    # True (duplicate 'l')
has_duplicate_characters("world")    # False
has_duplicate_characters("aabbcc")   # True
has_duplicate_characters("")         # False
```


## All Stream All Events

In [27]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01YADtCcjbgBt5gah5oZcu1b', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' comprehensive relational database containing 50,000+ fabric', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ated customer records with r

# Stream Only Text Events

In [34]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

final_message = stream.get_final_message()
# Get the complete message for database storage
print("\nFinal message:", final_message.content[0].text)

A comprehensive database containing fictional customer records, including made-up names, email addresses, phone numbers, and purchase histories used for software testing and development purposes.
Final message: A comprehensive database containing fictional customer records, including made-up names, email addresses, phone numbers, and purchase histories used for software testing and development purposes.


## Structured Outputs

In [38]:

messages = []
add_user_message(messages, "Generate a short event bridge rule as JSON")
answer = chat(messages)
print("Claude:", answer)

Claude: Here's a simple EventBridge rule in JSON format:

```json
{
  "Name": "EC2StateChangeRule",
  "Description": "Trigger on EC2 instance state changes",
  "State": "ENABLED",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running", "stopped"]
    }
  },
  "Targets": [
    {
      "Arn": "arn:aws:sns:us-east-1:123456789012:MyTopic",
      "Id": "1"
    }
  ]
}
```

This rule:
- Monitors EC2 instance state changes
- Triggers when instances transition to "running" or "stopped"
- Sends notifications to an SNS topic


In [ ]:
def chat(messages, system = None, temperature = 1.0, stop_sequences = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(
        **params
    )
    return message.content[0].text

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

sequence = ["```"]

text = chat(messages, stop_sequences=sequence)
print("Claude:", text)